# Food utils

Reusable building blocks for the iFood classification project: data
loading, the Lightning wrapper around any model, and evaluation
functions. This notebook does not contain any specific network
architecture, the architecture lives in each experiment notebook and
gets passed in from the outside.

In [ ]:
!pip install -q lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 842.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 20.5 MB/s eta 0:00:00


## Imports and reproducibility

In [ ]:
import random
import time
from pathlib import Path
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import lightning as L
import os
import json
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

## Image transforms

For training we use some basic augmentation, nothing extreme since color
matters a lot for food images. For validation we only resize and
normalize, no augmentation, since we want a stable and honest
measurement.

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def get_transforms(split, img_size=IMG_SIZE):
    if split == "train":
        return T.Compose([
            T.Resize(int(img_size * 1.15)),
            T.RandomCrop(img_size),
            T.RandomHorizontalFlip(p=0.5),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return T.Compose([
            T.Resize(int(img_size * 1.15)),
            T.CenterCrop(img_size),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])


def show_augmentations(img_path, n=8, img_size=IMG_SIZE):
    transform = get_transforms("train", img_size)
    img = Image.open(img_path).convert("RGB")

    fig, axes = plt.subplots(2, n // 2, figsize=(n * 2, 5))
    for ax in axes.flatten():
        t = transform(img).permute(1, 2, 0).numpy()
        t = t * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        t = np.clip(t, 0, 1)
        ax.imshow(t)
        ax.axis("off")
    plt.suptitle("Training augmentations", fontsize=13)
    plt.tight_layout()
    plt.show()

## Dataset class

Reads an image and its label from a dataframe with columns path and
label. If an image cannot be opened we skip it and try the next index
instead of returning a fake label, since a fake label would break the
loss function later.


In [ ]:
class FoodDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["path"]).convert("RGB")
        except (OSError, IOError):
            return self.__getitem__((idx + 1) % len(self.df))

        label = int(row["label"])
        if self.transform:
            img = self.transform(img)
        return img, label

## Data module

Loads the training labels csv, builds the full image path for each row,
applies an optional cap per class, and splits into train and validation.
Classes with less than 5 images go entirely into train so validation
never ends up with an empty class. Expects data_dir to contain
train_labels.csv and a train_set folder with the images inside.

In [ ]:
class FoodDataModule(L.LightningDataModule):
    def __init__(self, data_dir, img_size=IMG_SIZE, batch_size=64,
                 val_frac=0.15, cap=None, num_workers=2):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.img_size = img_size
        self.batch_size = batch_size
        self.val_frac = val_frac
        self.cap = cap
        self.num_workers = num_workers
        self.train_df = None
        self.val_df = None
        self.test_df = None
        self.num_classes = 251

    def _load_csv(self, csv_name, img_subfolder):
        csv_path = self.data_dir / csv_name
        df = pd.read_csv(csv_path)
        img_folder = self.data_dir / img_subfolder
        df["path"] = df["img_name"].apply(lambda x: str(img_folder / x))
        return df[["path", "label"]]

    def _load_annotations(self):
        return self._load_csv("train_labels.csv", "train_set")

    def _load_test_annotations(self):
        return self._load_csv("val_labels.csv", "val_set")

    def _apply_cap(self, df):
        if self.cap is None:
            return df
        groups = []
        for label, group in df.groupby("label"):
            if len(group) > self.cap:
                group = group.sample(self.cap, random_state=42)
            groups.append(group)
        return pd.concat(groups).reset_index(drop=True)

    def _stratified_split(self, df):
        from sklearn.model_selection import train_test_split
        counts = df["label"].value_counts()
        safe_labels = counts[counts >= 5].index
        df_safe = df[df["label"].isin(safe_labels)]
        df_unsafe = df[~df["label"].isin(safe_labels)]

        train_part, val_part = train_test_split(
            df_safe, test_size=self.val_frac,
            stratify=df_safe["label"], random_state=42
        )
        train_df = pd.concat([train_part, df_unsafe], ignore_index=True)
        return train_df, val_part.reset_index(drop=True)

    def setup(self, stage=None):
        df = self._load_annotations()
        df = self._apply_cap(df)
        self.train_df, self.val_df = self._stratified_split(df)
        self.test_df = self._load_test_annotations()

        print("train images:", len(self.train_df))
        print("val images:", len(self.val_df))
        print("test images:", len(self.test_df))

    def _make_loader(self, df, split):
        dataset = FoodDataset(df, transform=get_transforms(split, self.img_size))
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=(split == "train"),
            num_workers=self.num_workers,
        )

    def train_dataloader(self):
        return self._make_loader(self.train_df, "train")

    def val_dataloader(self):
        return self._make_loader(self.val_df, "val")

    def test_dataloader(self):
        return self._make_loader(self.test_df, "val")

## Lightning module

Wraps any model, the loss function, the optimizer and the training and
validation steps together. This part does not need to change when the
model architecture changes, since the model is passed in from the
outside.

In [ ]:
class FoodClassifier(L.LightningModule):
    def __init__(
        self,
        model,
        lr=1e-3,
        weight_decay=1e-4,
        optimizer_name="adam",
        scheduler_type="plateau",
        warmup_epochs=5,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch, stage):
        imgs, labels = batch
        logits = self(imgs)
        loss = self.criterion(logits, labels)
        acc = (logits.argmax(dim=1) == labels).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True)
        self.log(f"{stage}_acc", acc, prog_bar=True, on_epoch=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    @staticmethod
    def _split_params_for_weight_decay(model):
        decay, no_decay = [], []
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if param.ndim <= 1 or name.endswith(".bias"):
                no_decay.append(param)
            else:
                decay.append(param)
        return decay, no_decay

    def configure_optimizers(self):
        decay, no_decay = self._split_params_for_weight_decay(self.model)
        param_groups = [
            {"params": decay, "weight_decay": self.hparams.weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ]

        if self.hparams.optimizer_name == "adamw":
            optimizer = torch.optim.AdamW(param_groups, lr=self.hparams.lr)
        else:
            optimizer = torch.optim.Adam(param_groups, lr=self.hparams.lr)

        if self.hparams.scheduler_type == "plateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="max",
                factor=0.5,
                patience=3,
                min_lr=1e-6,
            )
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "monitor": "val_acc",
                    "interval": "epoch",
                    "frequency": 1,
                },
            }

        elif self.hparams.scheduler_type == "cosine_warmup":
            warmup_epochs = self.hparams.warmup_epochs
            total_epochs = self.trainer.max_epochs

            warmup = torch.optim.lr_scheduler.LinearLR(
                optimizer, start_factor=0.1, total_iters=warmup_epochs
            )
            cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=max(total_epochs - warmup_epochs, 1),
                eta_min=1e-6,
            )
            scheduler = torch.optim.lr_scheduler.SequentialLR(
                optimizer,
                schedulers=[warmup, cosine],
                milestones=[warmup_epochs],
            )
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch",
                    "frequency": 1,
                },
            }

        else:
            raise ValueError(f"Unknown scheduler_type: {self.hparams.scheduler_type}")

## Evaluation utilities

Functions used after training to run inference on the validation set,
compute metrics, and look at where the model is making mistakes.

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("trainable parameters:", f"{total:,}", f"({total / 1e6:.2f}M)")
    return total


def predict_all(model, dataloader, device="cuda"):
    model.eval().to(device)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)
    return torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy()


def evaluate(y_true, y_pred, class_names=None, verbose=True):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if verbose:
        for key, value in metrics.items():
            print(key, ":", round(value, 4))
        if class_names:
            print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    return metrics


def plot_confusion_matrix(y_true, y_pred, class_names=None, top_n=20):
    cm = confusion_matrix(y_true, y_pred)
    errors = cm.sum(axis=1) - cm.diagonal()
    top_classes = np.argsort(errors)[-top_n:]
    cm_sub = cm[np.ix_(top_classes, top_classes)]

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm_sub, ax=ax, cmap="Blues", fmt="d", annot=(top_n <= 20),
        xticklabels=[class_names[i] if class_names else i for i in top_classes],
        yticklabels=[class_names[i] if class_names else i for i in top_classes]
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix, top {top_n} most confused classes")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_training_curves(log_dir):
    import pandas as pd
    import matplotlib.pyplot as plt

    df = pd.read_csv(f"{log_dir}/metrics.csv")

    train_acc_col = "train_acc_epoch" if "train_acc_epoch" in df.columns else "train_acc"
    train_loss_col = "train_loss_epoch" if "train_loss_epoch" in df.columns else "train_loss"
    epoch_df = df.groupby("epoch").agg({
        train_acc_col: "max",
        train_loss_col: "max",
        "val_acc": "max",
        "val_loss": "max",
    }).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epoch_df["epoch"], epoch_df[train_acc_col], label="train")
    axes[0].plot(epoch_df["epoch"], epoch_df["val_acc"], label="val")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(epoch_df["epoch"], epoch_df[train_loss_col], label="train")
    axes[1].plot(epoch_df["epoch"], epoch_df["val_loss"], label="val")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def per_class_metrics(y_true, y_pred, class_names=None):
    report = classification_report(
        y_true, y_pred, output_dict=True, zero_division=0
    )
    rows = []
    labels_present = sorted(set(y_true) | set(y_pred))
    for label in labels_present:
        key = str(label)
        if key not in report:
            continue
        r = report[key]
        rows.append({
            "label": label,
            "class_name": class_names[label] if class_names else label,
            "precision": r["precision"],
            "recall": r["recall"],   # = accuracy per quella classe
            "f1": r["f1-score"],
            "n_samples": int(r["support"]),
        })
    df = pd.DataFrame(rows).sort_values("recall")
    return df


def plot_worst_classes(per_class_df, n=15):
    worst = per_class_df.head(n)
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(worst["class_name"], worst["recall"], color="salmon")
    ax.set_xlabel("Recall (accuracy per classe)")
    ax.set_title(f"Le {n} classi con performance peggiore sul test set")
    ax.invert_yaxis()
    for bar, n_samp in zip(bars, worst["n_samples"]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"n={n_samp}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

## Self supervised learning utilities

Shared helpers for the self supervised notebooks. None of these depend on a
specific pretext task, the encoder is always passed in from the outside, so
the same functions work for colorization, contrastive or any other pretext
task notebook.

### Feature extraction

Runs a frozen encoder over a dataloader and collects a pooled feature vector
per image together with its label. Global average pooling is applied on
whatever spatial map the encoder returns, the same pooling every supervised
net in this project already applies right before its classifier layer. The
encoder itself is never updated here, eval mode and no_grad make sure of
that.

In [ ]:
def extract_features(encoder, dataloader, device="cuda"):
    encoder.eval().to(device)
    gap = nn.AdaptiveAvgPool2d(1)

    all_feats, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            feats = encoder(inputs)
            feats = gap(feats).flatten(1)
            all_feats.append(feats.cpu())
            all_labels.append(labels)

    X = torch.cat(all_feats).numpy()
    y = torch.cat(all_labels).numpy()
    return X, y

### Linear probing and nearest neighbor probing

Two ways of judging the quality of features extracted by a self supervised
encoder. Linear probing trains a plain
logistic regression on top of the frozen features, nothing fancier, since
the point is to measure the features themselves and not the power of the
classifier sitting on top of them. The nearest neighbor probe is a cheaper
verification, if the features are any good, images from the same class
should already sit close to each other before any classifier is even
trained.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier


def run_linear_probe(X_train, y_train, X_val, y_val, class_names=None, max_iter=2000):
    # newer scikit-learn versions dropped the multi_class argument, lbfgs
    # already picks a multinomial fit on its own once there are more than
    # two classes, so nothing needs to be passed explicitly here
    clf = LogisticRegression(max_iter=max_iter)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    metrics = evaluate(y_val, y_pred, class_names=class_names)
    return metrics, clf


def run_knn_probe(X_train, y_train, X_val, y_val, class_names=None, n_neighbors=5):
    knn = KNeighborsClassifier(n_neighbors=n_neighbors)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_val)
    metrics = evaluate(y_val, y_pred, class_names=class_names)
    return metrics, knn

### Small housekeeping helpers

Loading the class name list and saving a metrics dictionary to disk both
show up identically in every experiment notebook, base net, medium net,
complex net, and now the self supervised ones. Kept here once instead of
copied everywhere.

In [ ]:
def load_class_names(data_dir):
    path = os.path.join(data_dir, "class_list.txt")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return [line.strip() for line in f.readlines()]


def save_metrics(metrics, log_dir, filename="metrics_summary.json"):
    os.makedirs(log_dir, exist_ok=True)
    path = os.path.join(log_dir, filename)
    with open(path, "w") as f:
        json.dump(metrics, f, indent=2)
    print("saved to", path)